## __4 Test & Evaluation__

## 4.1  __Import Libraries__

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

## 4.2 Define Transform

because we are in testing phase we want the model to view the photo in their original form without any flips or any augmentation we will just normalize them

In [ ]:
# Only normalization (no augmentation in testing)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), # تحويل الصور الى تدرجات الرمادي 
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 4.3 Define custom Dataset Class
تحويل المسارات للصور في مجلد ال سي اس في الى داتا سيت يمكن استخدامها من قبل pytorch

In [ ]:
class GestureDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

        # تحويل التسميات إلى أرقام
        self.label_map = {
            "Palm": 0,
            "Fist": 1
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]["image_path"]
        label = self.dataframe.iloc[idx]["label"]

        image = Image.open(img_path).convert("RGB")  # التأكد من وجود 3 تدرجات للون

        if self.transform:
            image = self.transform(image)             # تطبيق النورمالايزيشن

        label = self.label_map[label]                 # تحويل "Palm"/"Fist" إلى 0/1

        return image, label

## 4.4 Load test data


In [ ]:
test_df = pd.read_csv("test.csv")                 # قراءة ملف CSV الخاص بالاختبار
test_dataset = GestureDataset(test_df, transform=transform)  # إنشاء dataset
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)  # DataLoader
print("Test dataset size:", len(test_dataset))

## 4.5 Load best model 

Loading our Pretrained ResNet18 Model and Adapting It for Binary Classification


Important: Use updated ResNet syntax (it gives no warning)

In [ ]:
from torchvision.models import ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model with ImageNet weights
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

# Modify final layer
model.fc = nn.Linear(model.fc.in_features, 2)

# Load trained weights
model.load_state_dict(torch.load("best_model.pth", map_location=device))

model = model.to(device)
model.eval()

print("Model loaded successfully.")

## 4.6 Evaluate Model and calculating accuracy

In [ ]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

## 4.7 Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Palm", "Fist"],
            yticklabels=["Palm", "Fist"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 4.8 Classification Report

In [ ]:
report = classification_report(
    all_labels,
    all_preds,
    target_names=["Palm", "Fist"]
)

print("Classification Report:\n")
print(report)